In [23]:
import os
import pprint
import torch
from PIL import Image


pp = pprint.pprint

In [24]:
# Check MPS Available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using: MPS (Mac GPU) 🚀")
else:
    device = torch.device("cpu")
    print("Using: CPU")

Using: MPS (Mac GPU) 🚀


In [25]:
root_dir = 'dataset'

In [26]:
from torch.utils.data import Dataset

class BinDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        # Build class mapping from folder names
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir,d))
                               ])
        self.class_to_idx = {cls_name : idx for idx, cls_name in enumerate(self.classes)}

        
        # Scans all folders -> Finds all images -> Saves path + label
        for bin_folder in self.classes:
            bin_path = os.path.join(root_dir, bin_folder)
            label = self.class_to_idx[bin_folder]  # "cardboard" -> 0, "glass" -> 1, etc.
            
            for image_name in os.listdir(bin_path):
                if image_name.endswith('.jpg'):
                    image_path = os.path.join(bin_path, image_name)
                    self.samples.append((image_path, label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image ,label
    
dataset = BinDataset(
    root_dir=root_dir
)

pp(vars(dataset))
pp(len(dataset))


{'class_to_idx': {'cardboard': 0,
                  'glass': 1,
                  'metal': 2,
                  'paper': 3,
                  'plastic': 4,
                  'trash': 5},
 'classes': ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'],
 'root_dir': 'dataset',
 'samples': [('dataset/cardboard/cardboard_266.jpg', 0),
             ('dataset/cardboard/cardboard_272.jpg', 0),
             ('dataset/cardboard/cardboard_299.jpg', 0),
             ('dataset/cardboard/cardboard_058.jpg', 0),
             ('dataset/cardboard/cardboard_064.jpg', 0),
             ('dataset/cardboard/cardboard_070.jpg', 0),
             ('dataset/cardboard/cardboard_138.jpg', 0),
             ('dataset/cardboard/cardboard_110.jpg', 0),
             ('dataset/cardboard/cardboard_104.jpg', 0),
             ('dataset/cardboard/cardboard_312.jpg', 0),
             ('dataset/cardboard/cardboard_306.jpg', 0),
             ('dataset/cardboard/cardboard_307.jpg', 0),
             ('dataset/cardboar

In [27]:
from torchvision import transforms
from torch.utils.data import Subset, DataLoader
import numpy as np

# Training: with augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test: no augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets with different transforms
train_dataset_full = BinDataset(root_dir=root_dir, transform=train_transform)
val_dataset_full = BinDataset(root_dir=root_dir, transform=val_transform)

# Get indices and shuffle
indices = list(range(len(train_dataset_full)))
np.random.seed(42)
np.random.shuffle(indices)

# Split indices: 80% train, 10% val, 10% test
train_size = int(0.8 * len(indices))
val_size = int(0.1 * len(indices))

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

# Create subsets
train_dataset = Subset(train_dataset_full, train_indices)  # With augmentation
val_dataset = Subset(val_dataset_full, val_indices)        # No augmentation
test_dataset = Subset(val_dataset_full, test_indices)      # No augmentation

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train: 2021, Val: 252, Test: 254
Train batches: 64
Val batches: 8
Test batches: 8


In [34]:
# Setup ResNet18
from torchvision import models
import torch.nn as nn

# Load pretrained ResNet18
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 blocks of features (fine-tune them)
for param in model.features[-2:].parameters():
    param.requires_grad = True


# Replace final layer (ResNet18 has 512 features in fc layer)
num_classes = len(dataset.classes)  # 6 classes
# Replace classifier (EfficientNet uses 1280 features)
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Linear(256, num_classes)
)

# Move to device
model = model.to(device)

# Verify: only fc layer is trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} parameters")

Trainable: 1,458,870 / 4,337,026 parameters


In [48]:
# Loss function & Optimizer
import torch.optim as optim

# Loss function (For multi-class classification)
criterion = nn.CrossEntropyLoss()

# Optimizer (only train fc layer parameters)
optimizer = optim.Adam([
    {'params': model.features[-2:].parameters(), 'lr': 0.0001},  # Backbone: slower
    {'params': model.classifier.parameters(), 'lr': 0.001}       # Classifier: normal
])


In [49]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track stats
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100. * correct / total
    train_loss = running_loss / len(train_loader)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

Epoch [1/50] - Loss: 0.0726, Accuracy: 97.58%
Epoch [2/50] - Loss: 0.0436, Accuracy: 98.57%
Epoch [3/50] - Loss: 0.0497, Accuracy: 98.52%
Epoch [4/50] - Loss: 0.0308, Accuracy: 98.61%
Epoch [5/50] - Loss: 0.0494, Accuracy: 98.96%
Epoch [6/50] - Loss: 0.0425, Accuracy: 98.76%
Epoch [7/50] - Loss: 0.0263, Accuracy: 99.11%
Epoch [8/50] - Loss: 0.0279, Accuracy: 99.16%
Epoch [9/50] - Loss: 0.0179, Accuracy: 99.26%
Epoch [10/50] - Loss: 0.0196, Accuracy: 99.21%
Epoch [11/50] - Loss: 0.0231, Accuracy: 99.31%
Epoch [12/50] - Loss: 0.0417, Accuracy: 98.76%
Epoch [13/50] - Loss: 0.0233, Accuracy: 99.26%
Epoch [14/50] - Loss: 0.0571, Accuracy: 98.81%
Epoch [15/50] - Loss: 0.0251, Accuracy: 99.11%
Epoch [16/50] - Loss: 0.0428, Accuracy: 99.11%
Epoch [17/50] - Loss: 0.0309, Accuracy: 99.16%
Epoch [18/50] - Loss: 0.0278, Accuracy: 99.01%
Epoch [19/50] - Loss: 0.0196, Accuracy: 99.36%
Epoch [20/50] - Loss: 0.0292, Accuracy: 98.81%
Epoch [21/50] - Loss: 0.0321, Accuracy: 99.21%
Epoch [22/50] - Loss: 

In [50]:
# Validation
model.eval()
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        val_total += labels.size(0)
        val_correct += predicted.eq(labels).sum().item()

val_acc = 100. * val_correct / val_total
print(f"Validation Accuracy: {val_acc:.2f}%")

Validation Accuracy: 90.87%


In [51]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': dataset.classes
}, 'smartbin_model.pth')

print("Model saved!")
print(f"Classes: {dataset.classes}")

Model saved!
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
